In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [26]:
from pkgimp import *
from bson import ObjectId
from tqdm import tqdm

from nb2p import database, fileop, config, astparse
from nb2p.notebook import Notebook

Connect to database.

In [31]:
# MODEL_NAME = "Qwen2.5-Coder-7B-Instruct"
MODEL_NAME = "DeepSeek-Coder-V2-Lite-Instruct"

SETUP_NAME = "raw"
# SETUP_NAME = "raw_cot"
# SETUP_NAME = "raw_fewshot"

## Average on all data

In [32]:
def get_files(dataset_name: str):
    ground_truth_file = fileop.read_json(os.path.join(dataset_name, f"{dataset_name}_groundtruth.json"))
    result_json_files = sorted(glob.glob(f"{dataset_name}/{MODEL_NAME}/{SETUP_NAME}/*.json"), key=lambda x: int(x.split("/")[-1].split(".")[0]))
    # result_json_files = sorted(glob.glob(f"llm_proprietary/{dataset_name}/{SETUP_NAME}/*.json"), key=lambda x: int(x.split("/")[-1].split(".")[0]))

    return ground_truth_file, result_json_files

def load_result(json_files: List[str]):
    r = []
    
    for fpath in json_files:
        id = int(fpath.split("/")[-1].split(".")[0])
        with open(fpath) as f:
            js = json.load(f)
            r.append({
                'id': id,
                'gt': ground_truth_file[id]['segment_ends_ast'],
                'response_text': js['response'],
                'time': js['time']
            })

    return r

result = []

for ds in ['distilkaggle', 'pmbf']:
    ground_truth_file, result_json_files = get_files(ds)
    result.extend(load_result(result_json_files))

len(result)

2048

In [33]:
times = [r['time'] for r in result]
len(times), times[0]

(2048, 16.682095289230347)

In [34]:
np.mean(times), np.std(times)

(127.93561918463092, 74.22582966873821)

## Average on n_ast_children bins

In [1]:
def get_n_ast_children_region(index: int):
    return (index // 128 + 1) * 32
    
print(get_n_ast_children_region(0), 32)
print(get_n_ast_children_region(127), 32)
print(get_n_ast_children_region(128), 64)
print(get_n_ast_children_region(255), 64)
print(get_n_ast_children_region(1023), 256)

32 32
32 32
64 64
64 64
256 256
